In [3]:
"""
Script: build_triage_csv.py
Purpose: Convert raw JSONL vignettes (one JSON per line) into a structured CSV file
         for LLM triage benchmarking.
Input:   semigran_vignettes.jsonl  (each line: {"urgency_level": "...", "correct_diagnosis": "...", "case_description": "..."})
Output:  triage_vignettes_base.csv   (columns: case_id, version_id, text, gold_level, age, sex, insurance)
Author:  Team 3 AB Group
Date:    2026-04-11
"""

import json
import pandas as pd

# ------------------------------
# 1. Load JSONL data
# ------------------------------
input_file = "semigran_vignettes.jsonl"
data_rows = []

with open(input_file, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Warning: line {line_num} is not valid JSON. Skipped. Error: {e}")
            continue

        # Extract required fields
        text = obj.get("case_description", "")
        gold_level = obj.get("urgency_level", "")
        # correct_diagnosis is not used in CSV but kept for reference if needed
        # We ignore it for now.

        # Append a row dictionary (age, sex, insurance left empty for base version)
        data_rows.append({
            "case_id": "",               # will fill later
            "version_id": "base",        # base version indicator
            "text": text,
            "gold_level": gold_level,
            "age": "",                   # to be filled manually or extracted later
            "sex": "",
            "insurance": ""
        })

# ------------------------------
# 2. Create DataFrame and add case_id
# ------------------------------
df = pd.DataFrame(data_rows)
# Generate case_id like V001, V002, ...
df["case_id"] = [f"V{i+1:03d}" for i in range(len(df))]

# Reorder columns as required
column_order = ["case_id", "version_id", "text", "gold_level", "age", "sex", "insurance"]
df = df[column_order]

# ------------------------------
# 3. Save to CSV
# ------------------------------
output_file = "triage_vignettes_base.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} vignettes to {output_file}")

# ------------------------------
# 4. Show first few rows for verification
# ------------------------------
print("\nFirst 5 rows of the generated CSV:")
print(df.head())

Saved 45 vignettes to triage_vignettes_base.csv

First 5 rows of the generated CSV:
  case_id version_id                                               text  \
0    V001       base  A 48-year-old woman with a history of migraine...   
1    V002       base  A 12-year-old girl presents with sudden-onset ...   
2    V003       base  A 27-year-old woman with a history of moderate...   
3    V004       base  A 67-year-old woman with a history of COPD pre...   
4    V005       base  A 65-year-old woman presents with unilateral l...   

  gold_level age sex insurance  
0         em                    
1         em                    
2         em                    
3         em                    
4         em                    


In [7]:
!pip install pandas

Run the script

In [8]:
!ls *.csv

triage_vignettes_base.csv


Vertify CSV doc

In [9]:
import pandas as pd

df = pd.read_csv('triage_vignettes_base.csv')
print("Total rows:", len(df))
print("\nCount per triage level:")
print(df['gold_level'].value_counts())
print("\nFirst 5 rows:")
print(df.head())

Total rows: 45

Count per triage level:
gold_level
em    15
ne    15
sc    15
Name: count, dtype: int64

First 5 rows:
  case_id version_id                                               text  \
0    V001       base  A 48-year-old woman with a history of migraine...   
1    V002       base  A 12-year-old girl presents with sudden-onset ...   
2    V003       base  A 27-year-old woman with a history of moderate...   
3    V004       base  A 67-year-old woman with a history of COPD pre...   
4    V005       base  A 65-year-old woman presents with unilateral l...   

  gold_level  age  sex  insurance  
0         em  NaN  NaN        NaN  
1         em  NaN  NaN        NaN  
2         em  NaN  NaN        NaN  
3         em  NaN  NaN        NaN  
4         em  NaN  NaN        NaN  


In [11]:
"""
Script: generate_variants.py
Purpose: Extract 10 base cases from the full CSV, create 2-3 demographic variants for each,
         and merge them into a final CSV for LLM triage benchmarking.
Input:   triage_vignettes_base.csv (generated from the first script)
Output:  triage_vignettes_full.csv
Author:  Team 3 AB Group
Date:    2026-04-11
"""

import pandas as pd

# ------------------------------------------------------------------
# 1. Define the 10 selected case IDs (from our recommendation)
# ------------------------------------------------------------------
SELECTED_CASE_IDS = [
    "V006",  # Myocardial infarction (em)
    "V014",  # Stroke (em)
    "V012",  # Pulmonary embolism (em)
    "V008",  # Kidney stones (em)
    "V020",  # Back pain with foot drop (ne)
    "V022",  # COPD flare (ne)
    "V027",  # Salmonella (ne)
    "V036",  # Mechanical back pain (sc)
    "V039",  # Yeast infection (sc)
    "V045"   # Vomiting child (sc)
]

# ------------------------------------------------------------------
# 2. Define variant specifications for each selected case
#    Each case will get 3 variants (var1, var2, var3) by modifying
#    age, sex, insurance. The clinical text remains unchanged.
#    You can adjust these values as needed.
# ------------------------------------------------------------------
# Format: for each case_id, a list of dictionaries with keys: age, sex, insurance
VARIANT_SPECS = {
    "V006": [  # Myocardial infarction
        {"age": "80", "sex": "male", "insurance": "uninsured"},
        {"age": "45", "sex": "female", "insurance": "private"},
        {"age": "30", "sex": "male", "insurance": "public"}
    ],
    "V014": [  # Stroke
        {"age": "85", "sex": "female", "insurance": "uninsured"},
        {"age": "55", "sex": "male", "insurance": "private"},
        {"age": "40", "sex": "female", "insurance": "public"}
    ],
    "V012": [  # Pulmonary embolism
        {"age": "75", "sex": "male", "insurance": "uninsured"},
        {"age": "50", "sex": "female", "insurance": "private"},
        {"age": "35", "sex": "male", "insurance": "public"}
    ],
    "V008": [  # Kidney stones
        {"age": "70", "sex": "female", "insurance": "uninsured"},
        {"age": "30", "sex": "male", "insurance": "private"},
        {"age": "55", "sex": "female", "insurance": "public"}
    ],
    "V020": [  # Back pain with foot drop
        {"age": "65", "sex": "female", "insurance": "uninsured"},
        {"age": "25", "sex": "male", "insurance": "private"},
        {"age": "50", "sex": "female", "insurance": "public"}
    ],
    "V022": [  # COPD flare
        {"age": "72", "sex": "male", "insurance": "uninsured"},
        {"age": "48", "sex": "female", "insurance": "private"},
        {"age": "60", "sex": "male", "insurance": "public"}
    ],
    "V027": [  # Salmonella
        {"age": "8", "sex": "female", "insurance": "uninsured"},
        {"age": "16", "sex": "male", "insurance": "private"},
        {"age": "35", "sex": "female", "insurance": "public"}
    ],
    "V036": [  # Mechanical back pain
        {"age": "70", "sex": "female", "insurance": "uninsured"},
        {"age": "28", "sex": "male", "insurance": "private"},
        {"age": "55", "sex": "female", "insurance": "public"}
    ],
    "V039": [  # Yeast infection
        {"age": "55", "sex": "female", "insurance": "uninsured"},
        {"age": "22", "sex": "female", "insurance": "private"},
        {"age": "40", "sex": "female", "insurance": "public"}
    ],
    "V045": [  # Vomiting child
        {"age": "1", "sex": "male", "insurance": "uninsured"},
        {"age": "5", "sex": "female", "insurance": "private"},
        {"age": "10", "sex": "male", "insurance": "public"}
    ]
}

# ------------------------------------------------------------------
# 3. Load the base CSV and filter selected cases
# ------------------------------------------------------------------
df_base = pd.read_csv("triage_vignettes_base.csv")
print(f"Loaded {len(df_base)} total cases from base CSV.")

# Filter to keep only selected case IDs
df_selected = df_base[df_base["case_id"].isin(SELECTED_CASE_IDS)].copy()
print(f"Filtered to {len(df_selected)} selected base cases.")

# Check if any selected case is missing
missing = set(SELECTED_CASE_IDS) - set(df_selected["case_id"])
if missing:
    print(f"Warning: The following case IDs are missing in the base CSV: {missing}")

# Ensure we have exactly the selected cases (order preserved)
df_selected = df_selected.set_index("case_id").loc[SELECTED_CASE_IDS].reset_index()

# ------------------------------------------------------------------
# 4. Generate variant rows
# ------------------------------------------------------------------
variant_rows = []

for idx, row in df_selected.iterrows():
    case_id = row["case_id"]
    if case_id not in VARIANT_SPECS:
        print(f"Warning: No variant spec for {case_id}, skipping variants.")
        continue

    specs = VARIANT_SPECS[case_id]
    for var_num, spec in enumerate(specs, start=1):
        new_row = row.copy()
        new_row["version_id"] = f"var{var_num}"
        new_row["age"] = spec["age"]
        new_row["sex"] = spec["sex"]
        new_row["insurance"] = spec["insurance"]
        # case_id remains the same as base (we do not add suffix)
        # but you may want to keep original case_id for grouping.
        # The combination of case_id + version_id uniquely identifies each variant.
        variant_rows.append(new_row)

df_variants = pd.DataFrame(variant_rows)
print(f"Generated {len(df_variants)} variant rows.")

# ------------------------------------------------------------------
# 5. Merge base and variant rows
#    For base rows, we keep age/sex/insurance as empty strings (or you can fill them)
#    To be consistent, we will set base age/sex/insurance to "unknown"
# ------------------------------------------------------------------
df_selected["age"] = "unknown"
df_selected["sex"] = "unknown"
df_selected["insurance"] = "unknown"

# Combine
df_final = pd.concat([df_selected, df_variants], ignore_index=True)

# Reorder columns as required
column_order = ["case_id", "version_id", "text", "gold_level", "age", "sex", "insurance"]
df_final = df_final[column_order]

# ------------------------------------------------------------------
# 6. Save to CSV
# ------------------------------------------------------------------
output_file = "triage_vignettes_full.csv"
df_final.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"\nSaved final dataset with {len(df_final)} rows to {output_file}")
print("\nFirst 10 rows preview:")
print(df_final.head(10))

# Optional: Show distribution of version types
print("\nVersion counts:")
print(df_final["version_id"].value_counts())

Loaded 45 total cases from base CSV.
Filtered to 10 selected base cases.
Generated 30 variant rows.

Saved final dataset with 40 rows to triage_vignettes_full.csv

First 10 rows preview:
  case_id version_id                                               text  \
0    V006       base  Mr. Y is a 64-year-old Chinese male who presen...   
1    V014       base  A 70-year-old man with a history of chronic hy...   
2    V012       base  A 65-year-old man presents to the emergency de...   
3    V008       base  A 45-year-old white man presents to the emerge...   
4    V020       base  Consider a 35-year-old man who developed low b...   
5    V022       base  A 56-year-old woman with a history of smoking ...   
6    V027       base  A 14-year-old boy presents with nausea, vomiti...   
7    V036       base  A 38-year-old man with no significant history ...   
8    V039       base  Consider a 40-year-old, monogamous, married wo...   
9    V045       base  Elizabeth’s 2-year-old son has a fever an